# Sentimind: Run Full Pipeline (One Pass)
Notebook nay chay lai toan bo pipeline: preprocess -> train BiLSTM -> eval, sau do doc metrics cuoi.

In [1]:
from pathlib import Path
import json
import subprocess
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"Python = {sys.executable}")

ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
CONFIG_PRE = PROJECT_ROOT / "configs" / "preprocessing.yaml"
CONFIG_BILSTM = PROJECT_ROOT / "configs" / "bilstm.yaml"

for p in [ARTIFACTS_DIR, PROCESSED_DIR, CONFIG_PRE, CONFIG_BILSTM]:
    print("-", p)

PROJECT_ROOT = /home/sakana/Code/PTIT/NLP/sentimind
Python = /home/sakana/miniconda3/bin/python
- /home/sakana/Code/PTIT/NLP/sentimind/data/artifacts
- /home/sakana/Code/PTIT/NLP/sentimind/data/processed
- /home/sakana/Code/PTIT/NLP/sentimind/configs/preprocessing.yaml
- /home/sakana/Code/PTIT/NLP/sentimind/configs/bilstm.yaml


In [2]:
def run_cmd(cmd: list[str], cwd: Path):
    print("\n$ " + " ".join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd),
        text=True,
        capture_output=True,
        check=False,
    )
    if proc.stdout:
        print(proc.stdout)
    if proc.stderr:
        print(proc.stderr)
    if proc.returncode != 0:
        raise RuntimeError(f"Command failed with code {proc.returncode}: {' '.join(cmd)}")

print("Helper ready.")

Helper ready.


## 1) Preprocess
Sinh lai train/val/test va preprocessing_report.json.

In [3]:
run_cmd(["python", "scripts/preprocess.py", "--config", str(CONFIG_PRE)], PROJECT_ROOT)


$ python scripts/preprocess.py --config /home/sakana/Code/PTIT/NLP/sentimind/configs/preprocessing.yaml
16:42:39 | INFO     | __main__ | Loading raw data from data/raw/Combined Data.csv …
16:42:39 | INFO     | __main__ | Raw shape: (53043, 3)
16:42:39 | INFO     | src.data.preprocess | Dropped 362 null rows.
16:42:40 | INFO     | src.data.preprocess | Dropped 6 rows with text shorter than 3 chars after cleaning.
16:42:40 | INFO     | src.data.preprocess | Dropped 1630 duplicate (text, label) rows.
16:42:40 | INFO     | src.data.preprocess | Preprocessing done. 51045 / 53043 rows retained. Class distribution: {0: 16007, 1: 15093, 2: 3620, 3: 2501, 4: 894, 5: 2292, 6: 10638}
16:42:40 | INFO     | __main__ | Split sizes — train: 35731 | val: 7657 | test: 7657
16:42:40 | INFO     | __main__ | Splits saved to data/processed.
16:42:40 | INFO     | __main__ | Quality report saved to data/artifacts/preprocessing_report.json.
16:42:40 | INFO     | __main__ | ✓ All split files pass schema valid

## 2) Train BiLSTM
Cell nay co the chay lau tuy theo CPU/GPU.

In [4]:
run_cmd(["python", "scripts/train_bilstm.py", "--config", str(CONFIG_BILSTM)], PROJECT_ROOT)


$ python scripts/train_bilstm.py --config /home/sakana/Code/PTIT/NLP/sentimind/configs/bilstm.yaml
16:42:42 | INFO     | src.data.dataset | Loading existing vocabulary from data/artifacts/vocab.json.
16:42:42 | INFO     | src.data.dataset | Vocabulary loaded from data/artifacts/vocab.json (41363 tokens).
16:42:42 | INFO     | src.data.dataset | Dataloaders ready — train: 35731 | val: 7657 | test: 7657
16:42:42 | INFO     | __main__ | Model architecture:
BiLSTMClassifier(
  (embedding): Embedding(41363, 300, padding_idx=0)
  (embedding_dropout): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(300, 256, num_layers=2, batch_first=True, dropout=0.3, bidirectional=True)
  (classifier): Sequential(
    (0): Dropout(p=0.3, inplace=False)
    (1): Linear(in_features=512, out_features=7, bias=True)
  )
)
16:42:42 | INFO     | __main__ | Trainable parameters: 15,132,235
16:42:42 | INFO     | __main__ | Class weights: [0.17824631929397583, 0.1890440136194229, 0.7881807684898376, 1.1406339406967163,

## 3) Evaluate
Danh gia tren test split va sinh bilstm_metrics.json + confusion matrix.

In [5]:
run_cmd(["python", "scripts/eval_bilstm.py", "--config", str(CONFIG_BILSTM), "--split", "test"], PROJECT_ROOT)


$ python scripts/eval_bilstm.py --config /home/sakana/Code/PTIT/NLP/sentimind/configs/bilstm.yaml --split test

  BiLSTM Evaluation Results (test split)
  Accuracy  : 0.7583
  Macro F1  : 0.6936
  Weighted F1: 0.7615
  Normal       P=0.934 R=0.903 F1=0.918  n=2401
  Depression   P=0.743 R=0.673 F1=0.706  n=2264
  Anxiety      P=0.780 R=0.764 F1=0.772  n=543
  Bipolar      P=0.741 R=0.741 F1=0.741  n=375
  Personality Disorder P=0.408 R=0.582 F1=0.480  n=134
  Stress       P=0.501 R=0.651 F1=0.566  n=344
  Suicidal     P=0.644 R=0.701 F1=0.671  n=1596


16:48:59 | INFO     | src.data.dataset | Vocabulary loaded from data/artifacts/vocab.json (41363 tokens).
16:48:59 | INFO     | __main__ | Checkpoint loaded from data/artifacts/bilstm_best.pt.

  eval :  96%|█████████▌| 115/120 [00:01<00:00, 102.79it/s]
                                                           
16:49:01 | INFO     | src.utils.metrics | Metrics saved to data/artifacts/bilstm_metrics.json.
16:49:01 | INFO     | src.utils

## 4) Summary Metrics
Doc metrics de xem nhanh ket qua lan chay moi nhat.

In [6]:
metrics_path = ARTIFACTS_DIR / "bilstm_metrics.json"
if not metrics_path.exists():
    raise FileNotFoundError(metrics_path)

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("Model:", metrics.get("model"))
print("Split:", metrics.get("split"))
print("Accuracy:", metrics.get("accuracy"))
print("Macro F1:", metrics.get("macro_f1"))
print("Weighted F1:", metrics.get("weighted_f1"))

print("\nPer-class F1:")
for label, score in metrics.get("per_class", {}).items():
    print(f"- {label}: F1={score['f1']:.4f}, support={score['support']}")

Model: bilstm
Split: test
Accuracy: 0.7583
Macro F1: 0.6936
Weighted F1: 0.7615

Per-class F1:
- Normal: F1=0.9181, support=2401
- Depression: F1=0.7062, support=2264
- Anxiety: F1=0.7721, support=543
- Bipolar: F1=0.7413, support=375
- Personality Disorder: F1=0.4800, support=134
- Stress: F1=0.5664, support=344
- Suicidal: F1=0.6713, support=1596
